# 04 - Feature Engineering

## Objective

The objective of this notebook is to prepare the cleaned HR dataset
for machine learning by creating meaningful features, transforming
the target variable, and separating machine learning features from
identifiers and unnecessary columns.

The main tasks include:

1. Load the cleaned dataset.
2. Remove EDA-only columns.
3. Create meaningful derived features.
4. Encode the target variable.
5. Identify numerical and categorical features.
6. Prepare the feature matrix and target variable.
7. Save the feature-engineered dataset.

In [1]:
# Importing libaries:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

## 2. Loading Cleaned Dataset

Load the cleaned dataset generated during the data cleaning stage.

In [2]:
df = pd.read_csv('cleaned_hr_data.csv')

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,4,1,6,3,3,2,2,2,2


In [3]:
df.shape

(1470, 32)

In [4]:
df.columns.tolist()

['Age',
 'Attrition',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [5]:
columns_to_remove = [
    'TenureGroup',
    'Attrition_Encoded'
]

df.drop(
    columns=columns_to_remove,
    errors='ignore',
    inplace=True
)

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,4,1,6,3,3,2,2,2,2


### Observation

Columns created specifically for exploratory analysis are removed
from the feature-engineering dataset when they are not required
for machine learning.

## 3. Encoding Target Variable

The target variable `Attrition` represents whether an employee
left the organization.

For machine learning, the categorical target is converted into
a binary numerical representation:

- Yes = 1
- No = 0

In [6]:
df['Attrition'] = df['Attrition'].map({
    'No': 0,
    'Yes': 1
})

df['Attrition'].value_counts()

Attrition
0    1233
1     237
Name: count, dtype: int64

In [7]:
df['Attrition'].unique()

array([1, 0])

## 4. Creating Derived Features

### 4.1 Age Group

Employees are grouped into broad age categories to support
HR-level analysis.

In [8]:
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 25, 35, 45, 60],
    labels=[
        '18-25',
        '26-35',
        '36-45',
        '46-60'
    ]
)

df['AgeGroup'].value_counts()

AgeGroup
26-35    606
36-45    468
46-60    273
18-25    123
Name: count, dtype: int64

### 4.2 Income Group

Monthly income is converted into broad income groups.

This feature can help analyze whether employees in different
salary ranges have different attrition patterns.

In [9]:
df['IncomeGroup'] = pd.qcut(
    df['MonthlyIncome'],
    q=4,
    labels=[
        'Low',
        'Medium',
        'High',
        'Very High'
    ]
)

df['IncomeGroup'].value_counts()

IncomeGroup
Low          369
Very High    368
High         367
Medium       366
Name: count, dtype: int64

### 4.3 Company Tenure Group

Employees are grouped according to their years at the company.

This feature provides a categorical representation of employee tenure.

In [10]:
df['TenureGroup'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 20, np.inf],
    labels=[
        '0-2 Years',
        '3-5 Years',
        '6-10 Years',
        '11-20 Years',
        '20+ Years'
    ]
)

df['TenureGroup'].value_counts()

TenureGroup
6-10 Years     448
3-5 Years      434
0-2 Years      342
11-20 Years    180
20+ Years       66
Name: count, dtype: int64

### 4.4 Promotion Status

A binary feature is created to indicate whether an employee
received a promotion within the last two years.

A value of 1 indicates a recent promotion, while 0 indicates
that the employee has not been promoted recently.

In [11]:
df['RecentlyPromoted'] = np.where(
    df['YearsSinceLastPromotion'] <= 2,
    1,
    0
)

df['RecentlyPromoted'].value_counts()

RecentlyPromoted
1    1097
0     373
Name: count, dtype: int64

### 4.5 Job Level Group

Job levels are grouped into broader categories to simplify
organizational-level analysis.

In [12]:
df['JobLevelGroup'] = pd.cut(
    df['JobLevel'],
    bins=[0, 1, 3, 5],
    labels=[
        'Entry Level',
        'Mid Level',
        'Senior Level'
    ]
)

df['JobLevelGroup'].value_counts()

JobLevelGroup
Mid Level       752
Entry Level     543
Senior Level    175
Name: count, dtype: int64

### 4.6 Overall Satisfaction Score

The dataset contains multiple satisfaction-related variables.

An overall satisfaction score is created by calculating the average
of:

- Environment Satisfaction
- Job Satisfaction
- Relationship Satisfaction
- Work-Life Balance

In [13]:
satisfaction_columns = [
    'EnvironmentSatisfaction',
    'JobSatisfaction',
    'RelationshipSatisfaction',
    'WorkLifeBalance'
]

df['OverallSatisfaction'] = df[satisfaction_columns].mean(axis=1)

df['OverallSatisfaction'].describe()

count    1470.000000
mean        2.730952
std         0.505815
min         1.000000
25%         2.500000
50%         2.750000
75%         3.000000
max         4.000000
Name: OverallSatisfaction, dtype: float64

### 4.7 Company Experience Ratio

The company experience ratio represents the proportion of an
employee's total working experience spent at the current company.

This feature may help identify differences between relatively
new employees and long-term employees.

In [14]:
df['CompanyExperienceRatio'] = np.where(
    df['TotalWorkingYears'] > 0,
    df['YearsAtCompany'] / df['TotalWorkingYears'],
    0
)

df['CompanyExperienceRatio'].describe()

count    1470.000000
mean        0.678074
std         0.328093
min         0.000000
25%         0.411765
50%         0.800000
75%         1.000000
max         1.000000
Name: CompanyExperienceRatio, dtype: float64

In [15]:
new_features = [
    'AgeGroup',
    'IncomeGroup',
    'TenureGroup',
    'RecentlyPromoted',
    'JobLevelGroup',
    'OverallSatisfaction',
    'CompanyExperienceRatio'
]

df[new_features].head(10)

,AgeGroup,IncomeGroup,TenureGroup,RecentlyPromoted,JobLevelGroup,OverallSatisfaction,CompanyExperienceRatio
0,36-45,High,6-10 Years,1,Mid Level,2.00,0.750000
1,46-60,High,6-10 Years,1,Mid Level,3.00,1.000000
2,36-45,Low,0-2 Years,1,Entry Level,3.00,0.000000
3,26-35,Low,6-10 Years,0,Entry Level,3.25,1.000000
4,26-35,Medium,0-2 Years,1,Entry Level,2.50,0.333333
5,26-35,Medium,6-10 Years,0,Entry Level,3.25,0.875000
6,46-60,Low,0-2 Years,1,Entry Level,1.75,0.083333
7,26-35,Low,0-2 Years,1,Entry Level,3.00,1.000000
8,36-45,Very High,6-10 Years,1,Mid Level,3.00,0.900000
9,36-45,High,6-10 Years,0,Mid Level,2.50,0.411765


### checking for invalid values 

In [16]:
df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithCurrManager        0
AgeGroup                    0
IncomeGrou

In [18]:
df[new_features].isnull().sum()

AgeGroup                  0
IncomeGroup               0
TenureGroup               0
RecentlyPromoted          0
JobLevelGroup             0
OverallSatisfaction       0
CompanyExperienceRatio    0
dtype: int64

## 5. Identifying Feature Types

Machine learning preprocessing will treat numerical and categorical
variables differently.

Numerical features can be scaled where required, while categorical
features will be encoded using one-hot encoding.

In [19]:
X = df.drop(columns=['Attrition'])

y = df['Attrition']

In [20]:
numerical_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

In [21]:
print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'RecentlyPromoted', 'OverallSatisfaction', 'CompanyExperienceRatio']

Categorical Features:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'AgeGroup', 'IncomeGroup', 'TenureGroup', 'JobLevelGroup']


In [22]:
if 'EmployeeNumber' in numerical_features:
    numerical_features.remove('EmployeeNumber')

X = X.drop(columns=['EmployeeNumber'])

In [23]:
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 26
Number of categorical features: 11


In [24]:
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True) * 100)

Attrition
0    1233
1     237
Name: count, dtype: int64

Target percentage:
Attrition
0    83.877551
1    16.122449
Name: proportion, dtype: float64


## 6. Saving Feature-Engineered Dataset

The feature-engineered dataset will be saved for use in the
machine learning notebook.

The dataset contains the original cleaned features along with
selected derived features.

In [26]:
df.to_csv(
    'feature_engineered_hr_data.csv',
    index=False
)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!
